In [1]:
import os
print(os.cpu_count())

12


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lab-1") \
        .master("local[11]") \
        .config("spark.eventLog.enabled", "false") \
        .getOrCreate()

In [3]:
s = "DEST_COUNTRY_NAME STRING,ORIGIN_COUNTRY_NAME STRING,count INTEGER"

In [4]:
df_csv = spark.readStream.format('csv') \
            .option('path', 'file:///home/itversity/spark/lecture/csv/') \
            .schema(s) \
            .option('header', True) \
            .load()

In [5]:
from pyspark.sql.functions import sum

df_agg = df_csv.groupBy('DEST_COUNTRY_NAME','ORIGIN_COUNTRY_NAME').agg(sum("count"))

In [6]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [7]:
spark.conf.set("spark.sql.shuffle.partitions", 11)

In [8]:
spark.conf.get("spark.sql.shuffle.partitions")

'11'

In [9]:

output = df_agg.writeStream.format('console') \
        .outputMode('update') \
        .trigger(processingTime = '5 seconds') \
        .option('truncate', 'false') 

In [10]:
query = output.start()

In [11]:
query.stop()